# MESACLIP + FOSI Sea Ice Area Validation - Timeseries
## Compare regional sea ice area (SIA) from MESACLIP and FOSI against NSIDC CDR obs, over the EngressNet ML regional domain

Adapted from `mesaclip_validate_sic_timeseries.ipynb` to (1) add FOSI alongside MESACLIP and (2) integrate SIA over
the actual ML regional domain (`DEFAULT_BBOX` in `functions_engressnet.py`: lat 60-80N, lon 170-220E) instead of the
whole Northern Hemisphere.

In [ ]:
import numpy as np
import xarray as xr
from glob import glob
import matplotlib.pyplot as plt
import matplotlib as mpl

# Presentation-friendly styling + a fixed, colorblind-safe categorical palette (validated with
# the dataviz skill's palette validator: adjacent-pair CVD dE >= 8, normal-vision dE >= 15)
# applied consistently to every figure in this notebook and its companion
# `mesaclip_fosi_validate_sic_spatial.ipynb`, so MESACLIP/FOSI/CDR mean the same color everywhere.
PALETTE = {
    "MESACLIP": "#2a78d6",
    "FOSI": "#eb6834",
    "CDR": "#1baf7a",
    "PIOMAS": "#eda100",
    "members": "#c9c8c2",
}
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 13,
    "axes.edgecolor": "#898781",
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.8,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "legend.fontsize": 11,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 220,
})


## Regional domain
- must match the ML training/eval domain -- see `Version4/functions_engressnet.py:48` (`DEFAULT_BBOX`)

In [ ]:
# lon in degrees_east, lat in degrees_north -- mirrors functions_engressnet.DEFAULT_BBOX
BBOX = {"lon_min": -190, "lon_max": -140, "lat_min": 60, "lat_max": 80}

def bbox_ij_slice(tlat, tlon, bbox):
    """Index-space (j, i) bounding box enclosing bbox on a curvilinear (2D lat/lon) grid.
    Longitude is normalized to [0, 360) on both sides of the comparison so a bbox crossing
    the dateline (like this one, 170E-220E) works the same as functions_engressnet.py's masking."""
    lon_min = bbox["lon_min"] % 360
    lon_max = bbox["lon_max"] % 360
    tlon360 = tlon % 360
    mask = (tlat >= bbox["lat_min"]) & (tlat <= bbox["lat_max"]) & (tlon360 >= lon_min) & (tlon360 <= lon_max)
    jj, ii = np.where(mask.values if hasattr(mask, "values") else mask)
    return slice(int(jj.min()), int(jj.max()) + 1), slice(int(ii.min()), int(ii.max()) + 1)

def fill_nan_coord(da):
    """MESACLIP/FOSI set TLAT/TLON themselves to NaN over all-land computational blocks
    (not just the data), which pcolormesh/contour reject as coordinate input. Since aice is
    NaN at exactly those same cells already (real land there, nothing to plot), it's safe to
    fill the coordinate gaps from nearby valid neighbors -- the mesh cell lands in roughly the
    right place but stays uncolored (NaN data), same as if TLAT/TLON had never been masked."""
    dims = da.dims
    return da.ffill(dims[0]).bfill(dims[0]).ffill(dims[1]).bfill(dims[1])

## Load MESACLIP data
- CESM-HR (ne120_t12) 9-member ensemble, monthly `aice` (%)
- historical 1920-2005 (`/gdex/data/d651007`) spliced with RCP8.5 2006-2100 (`/gdex/data/d651009`), matched by ensemble suffix
- pulled raw rather than from the old duvivier-preprocessed file, which lives in a `cgd`-group-only
  directory (`/glade/campaign/cgd/ppc/duvivier/...`) this account no longer has read access to

In [ ]:
dir_hist = "/gdex/data/d651007"
dir_rcp85 = "/gdex/data/d651009"

hist_dirs = sorted(glob(f"{dir_hist}/*ihesp-hires*"))
rcp85_dirs = sorted(glob(f"{dir_rcp85}/*ihesp-hires*"))
# both dir names end in the same 3-digit ensemble suffix (e.g. "...002") -- match on that
member_ids = [d[-3:] for d in hist_dirs]
assert member_ids == [d[-3:] for d in rcp85_dirs], "historical/RCP8.5 member ordering does not match"
print("MESACLIP members:", member_ids)

In [ ]:
# open each member (historical + RCP8.5), cropping to the regional bbox via `preprocess=` so the
# crop happens on each file as it's opened -- BEFORE combine="by_coords" builds a dask graph over
# the full global (nj=2400, ni=3600) grid. The previous approach (isel() *after* opening the full
# mfdataset) left every file's aice dask array keyed to the full-grid chunk structure even though
# the isel output was small; reading even a single member that way pushed memory past 20+ GB and
# crashed the kernel. Cropping first (confirmed empirically before applying this fix) brings a
# single member's full-record read down to ~4.4 GB.
mesa_members = []

# crop indices only need computing once -- all members share the same native ne120_t12 grid
_ds0 = xr.open_dataset(sorted(glob(f"{hist_dirs[0]}/ice/proc/tseries/month_1/*.aice.*.nc"))[0])
nj_sl_mesa, ni_sl_mesa = bbox_ij_slice(_ds0.TLAT, _ds0.TLON, BBOX)
_ds0.close()

def crop_mesa_aice(ds):
    return ds[["aice", "tarea", "TLAT", "TLON"]].isel(nj=nj_sl_mesa, ni=ni_sl_mesa)

for hdir, rdir in zip(hist_dirs, rcp85_dirs):
    hist_files = sorted(glob(f"{hdir}/ice/proc/tseries/month_1/*.aice.*.nc"))
    rcp85_files = sorted(glob(f"{rdir}/ice/proc/tseries/month_1/*.aice.*.nc"))
    ds_h = xr.open_mfdataset(hist_files, combine="by_coords", preprocess=crop_mesa_aice,
                                 data_vars="minimal", compat="override", coords="minimal")
    ds_r = xr.open_mfdataset(rcp85_files, combine="by_coords", preprocess=crop_mesa_aice,
                                 data_vars="minimal", compat="override", coords="minimal")
    mesa_members.append(xr.concat([ds_h, ds_r], dim="time"))

ds_mesaclip = xr.concat(mesa_members, dim="member_id")
ds_mesaclip = ds_mesaclip.assign_coords(member_id=member_ids)
# TLAT/TLON are identical across members, so xr.concat's default compat check
# leaves them without a member_id dim at all (unlike aice, which does vary) --
# use them directly rather than trying to isel(member_id=0)
# also fill the NaN'd-out land-block coordinates so pcolormesh/contour can use them (see fill_nan_coord)
mesa_lat2d = fill_nan_coord(ds_mesaclip.TLAT)
mesa_lon2d = fill_nan_coord(ds_mesaclip.TLON)
ds_mesaclip


## Load FOSI data
- CESM-HR (t13), JRA55-forced, single realization, monthly `aice` (fraction 0-1)
- same native tripole grid as MESACLIP (confirmed identical TLAT/TLON), so shares the same crop indices

In [ ]:
dir_fosi = "/glade/campaign/cgd/oce/projects/FOSI_BGC/HR/g.e22.TL319_t13.G1850ECOIAF_JRA_HR.4p2z.001/ice/proc/tseries/month_1"
fosi_files = sorted(glob(f"{dir_fosi}/*.aice.*.nc"))

# same preprocess-crop fix as the MESACLIP load above -- crop before combine="by_coords" builds
# its dask graph, instead of isel()-ing the full global grid after opening
_ds0 = xr.open_dataset(fosi_files[0])
nj_sl_fosi, ni_sl_fosi = bbox_ij_slice(_ds0.TLAT, _ds0.TLON, BBOX)
_ds0.close()

def crop_fosi_aice(ds):
    return ds[["aice", "tarea", "TLAT", "TLON"]].isel(nj=nj_sl_fosi, ni=ni_sl_fosi)

ds_fosi = xr.open_mfdataset(fosi_files, combine="by_coords", preprocess=crop_fosi_aice,
                                data_vars="minimal", compat="override", coords="minimal")
fosi_lat2d = fill_nan_coord(ds_fosi.TLAT)
fosi_lon2d = fill_nan_coord(ds_fosi.TLON)
ds_fosi


## Load CDR satellite SIC data
- daily NH data on 25km x 25km EASE grid
- crop to the bbox *before* resampling to monthly -- doing it after means averaging ~50x more
  data than needed and is dramatically slower

In [ ]:
dir_grid = "/glade/campaign/cesm/development/pcwg/ssmi/"
ds_grid = xr.open_mfdataset(dir_grid + "grid_nh.nc", decode_times=False)
nj_sl_cdr, ni_sl_cdr = bbox_ij_slice(ds_grid.latitude, ds_grid.longitude, BBOX)

cdr_lat2d = ds_grid.latitude.isel(ygrid=nj_sl_cdr, xgrid=ni_sl_cdr)
cdr_lon2d = ds_grid.longitude.isel(ygrid=nj_sl_cdr, xgrid=ni_sl_cdr)

In [ ]:
dir_in = "/glade/campaign/cesm/development/pcwg/ssmi/CDR/"
file_in = "cdr_seaice_conc_daily_nh.cdr.noleap.19790101-20211231.nc"
ds_cdr = xr.open_dataset(dir_in + file_in, chunks={"time": 365})

# reassign a proper noleap cftime time axis (the raw file's time coordinate isn't usable directly)
dates_noleap = xr.date_range(start="1979-01-01", end="2021-12-31", freq="D", calendar="noleap", use_cftime=True)
ds_cdr["time"] = dates_noleap

# grid_nh.nc's (ygrid, xgrid) and the data file's (jdim, idim) are the same native EASE grid
# in the same index order, just labeled differently -- so the bounding box computed above
# from the grid file applies directly here
sic_cdr = ds_cdr.cdr_seaice_conc_daily.isel(jdim=nj_sl_cdr, idim=ni_sl_cdr)

## Compute regional sea ice area (SIA)
- SIA = sum over the cropped domain of (ice concentration fraction x grid cell area)
- MESACLIP `aice` is in %, FOSI `aice` is a 0-1 fraction, CDR is in % -- normalize all to fraction before integrating
- `tarea` (MESACLIP/FOSI, native grid) is in m^2; CDR cells are a fixed 25km x 25km

In [ ]:
# MESACLIP: build SIA lazily per member (not on the full 9-member concatenated ds_mesaclip) --
# see cell below ("this cell does all the actual data reading...") for why: computing on the
# full concat was materializing enough raw aice/tarea data at once to crash the kernel.
def member_sia(ds_member):
    # aice is %, tarea in m^2 -> SIA in km^2
    return ((ds_member.aice / 100) * ds_member.tarea).sum(dim=["nj", "ni"]) / 1e6

sia_mesaclip_members_lazy = [member_sia(m) for m in mesa_members]

# FOSI: aice is already a 0-1 fraction; single realization, cheap to keep as one lazy array
sia_fosi = (ds_fosi.aice * ds_fosi.tarea).sum(dim=["nj", "ni"]) / 1e6


In [ ]:
sic_cdr_monthly = sic_cdr.resample(time="ME").mean()
# CDR is %; each cell is a fixed 25km x 25km EASE grid cell -> SIA in km^2
sia_cdr = ((sic_cdr_monthly / 100) * (25 * 25)).sum(dim=["jdim", "idim"])

### Get annual mean, March, and September

In [ ]:
# MESACLIP -- same month-selection / annual-resample as before, applied lazily per member
def month_select_and_annual(da):
    da_mar = da.sel(time=da.time.dt.month == 3)
    da_sep = da.sel(time=da.time.dt.month == 9)
    da_ann = da.resample(time="YE").mean()
    return da_ann, da_mar, da_sep

sia_mesaclip_members_lazy = [month_select_and_annual(m) for m in sia_mesaclip_members_lazy]

# FOSI
sia_fosi_ann, sia_fosi_mar, sia_fosi_sep = month_select_and_annual(sia_fosi)

# CDR
sia_cdr_ann, sia_cdr_mar, sia_cdr_sep = month_select_and_annual(sia_cdr)


In [ ]:
# this cell does all the actual data reading for the full time series (all months, all years).
# MESACLIP members are computed and concatenated one at a time: computing the full 9-member
# concatenated array in one `.compute()` call was materializing enough raw aice/tarea data
# simultaneously to crash the kernel (VS Code "kernel crashed", no traceback -- consistent with
# an OOM kill under this environment's memory pressure). Looping bounds peak memory to ~1 member
# at a time instead of all 9, and prints progress so a slow run doesn't look hung before it dies.
import gc

ann_list, mar_list, sep_list = [], [], []
for i, (mid, (da_ann, da_mar, da_sep)) in enumerate(zip(member_ids, sia_mesaclip_members_lazy)):
    print(f"MESACLIP SIA: member {mid} ({i + 1}/{len(member_ids)})...", flush=True)
    ann_list.append(da_ann.compute())
    mar_list.append(da_mar.compute())
    sep_list.append(da_sep.compute())
    gc.collect()

sia_mesaclip_ann = xr.concat(ann_list, dim="member_id").assign_coords(member_id=member_ids)
sia_mesaclip_mar = xr.concat(mar_list, dim="member_id").assign_coords(member_id=member_ids)
sia_mesaclip_sep = xr.concat(sep_list, dim="member_id").assign_coords(member_id=member_ids)
del ann_list, mar_list, sep_list
gc.collect()

sia_fosi_ann, sia_fosi_mar, sia_fosi_sep = (
    x.compute() for x in (sia_fosi_ann, sia_fosi_mar, sia_fosi_sep)
)
sia_cdr_ann, sia_cdr_mar, sia_cdr_sep = (
    x.compute() for x in (sia_cdr_ann, sia_cdr_mar, sia_cdr_sep)
)
print("done.")


## Plot timeseries to compare

In [ ]:
import os
dir_out = "/glade/work/skygale/_projects/SeaIceDownscaling/Version5/saved_figs/mesaclip_fosi_validation/"
os.makedirs(dir_out, exist_ok=True)

In [ ]:
nens = len(sia_mesaclip_ann.member_id)
print("MESACLIP ensemble members:", nens)

# convert all to millions of km^2
sia_mesaclip_ann_m = sia_mesaclip_ann / 1e6
sia_mesaclip_mar_m = sia_mesaclip_mar / 1e6
sia_mesaclip_sep_m = sia_mesaclip_sep / 1e6

sia_fosi_ann_m = sia_fosi_ann / 1e6
sia_fosi_mar_m = sia_fosi_mar / 1e6
sia_fosi_sep_m = sia_fosi_sep / 1e6

sia_cdr_ann_m = sia_cdr_ann / 1e6
sia_cdr_mar_m = sia_cdr_mar / 1e6
sia_cdr_sep_m = sia_cdr_sep / 1e6

In [ ]:
def plot_panel(ax, mesa_da, fosi_da, cdr_da, title):
    # x-values are pulled from each series' own time coord (not a single shared array) --
    # annual resampling and month-selection don't always produce the same-length series
    # (e.g. CDR has a real multi-month sensor gap in 1987-88), so a shared x array can
    # silently mismatch a given y series' length
    xarr_mesa = mesa_da.time.dt.year.values
    xarr_fosi = fosi_da.time.dt.year.values
    xarr_cdr = cdr_da.time.dt.year.values
    for ii in range(nens):
        ax.plot(xarr_mesa, mesa_da.isel(member_id=ii), color=PALETTE["members"],
                 linewidth=1, label="_nolegend_", zorder=1)
    ax.plot(xarr_mesa, mesa_da.mean(dim="member_id"), label=f"MESACLIP ensemble mean (n={nens})",
             color=PALETTE["MESACLIP"], linewidth=2.2, zorder=3)
    ax.plot(xarr_fosi, fosi_da, label="FOSI", color=PALETTE["FOSI"], linewidth=2.2, zorder=3)
    ax.plot(xarr_cdr, cdr_da, label="NSIDC CDR", color=PALETTE["CDR"], linewidth=2.2, zorder=3)
    # historical -> RCP8.5 forcing splice: the one genuinely meaningful vertical reference in
    # this series (replaces 3 unlabeled decade gridlines at 1950/2000/2050 that didn't mark
    # anything real -- flagged in case that wasn't the intent)
    ax.axvline(x=2006, color="#898781", linestyle="--", linewidth=1.2, zorder=2, label="_nolegend_")
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_xlim([1920, 2100])
    ax.set_ylabel("Million km$^2$")


In [ ]:
# annual panel spans the full width, march/september share the row below --
# avoids the empty 4th quadrant a plain 2x2 grid left, and constrained_layout
# keeps the panels tight instead of the default wide gutters
fig = plt.figure(figsize=(13, 9), constrained_layout=True)
fout = "mesaclip_fosi_regional_timeseries_annual_mean"
gs = fig.add_gridspec(2, 2)

ax = fig.add_subplot(gs[0, :])
plot_panel(ax, sia_mesaclip_ann_m, sia_fosi_ann_m, sia_cdr_ann_m, "Annual Mean Regional SIA")
ax.legend(loc="lower left", ncol=1)

ax = fig.add_subplot(gs[1, 0])
plot_panel(ax, sia_mesaclip_mar_m, sia_fosi_mar_m, sia_cdr_mar_m, "March Regional SIA")

ax = fig.add_subplot(gs[1, 1])
plot_panel(ax, sia_mesaclip_sep_m, sia_fosi_sep_m, sia_cdr_sep_m, "September Regional SIA")

fig.savefig(dir_out + fout + ".png", bbox_inches="tight", dpi=220)


## Load sea ice thickness (for sea ice volume)

Sea ice **volume** (SIV) isn't output directly by MESACLIP or FOSI, so it's derived here from
monthly ice thickness (`hi`) and grid-cell area (`tarea`, already loaded above). CICE's `hi` is
documented (in each file's own variable attrs) as the **grid-cell-mean** thickness --
`comment: "ice volume per unit grid cell area"` -- so `SIV = sum(hi * tarea)` over the cropped
domain directly; no extra multiplication by `aice` is needed (that would double-count
concentration, since `hi` already reflects it).

In [ ]:
# reuse the same members / regional crop indices (nj_sl_mesa, ni_sl_mesa) established above for
# aice. Also crops via `preprocess=` at open time (same fix as the aice load above) instead of
# isel()-ing after opening the full global grid.
def crop_mesa_hi(ds):
    return ds[["hi"]].isel(nj=nj_sl_mesa, ni=ni_sl_mesa)

mesa_hi_members = []
for hdir, rdir in zip(hist_dirs, rcp85_dirs):
    hist_files = sorted(glob(f"{hdir}/ice/proc/tseries/month_1/*.hi.*.nc"))
    rcp85_files = sorted(glob(f"{rdir}/ice/proc/tseries/month_1/*.hi.*.nc"))
    ds_h = xr.open_mfdataset(hist_files, combine="by_coords", preprocess=crop_mesa_hi,
                                 data_vars="minimal", compat="override", coords="minimal")
    ds_r = xr.open_mfdataset(rcp85_files, combine="by_coords", preprocess=crop_mesa_hi,
                                 data_vars="minimal", compat="override", coords="minimal")
    mesa_hi_members.append(xr.concat([ds_h, ds_r], dim="time"))

ds_mesaclip_hi = xr.concat(mesa_hi_members, dim="member_id").assign_coords(member_id=member_ids)
ds_mesaclip_hi


In [ ]:
def crop_fosi_hi(ds):
    return ds[["hi"]].isel(nj=nj_sl_fosi, ni=ni_sl_fosi)

fosi_hi_files = sorted(glob(f"{dir_fosi}/*.hi.*.nc"))
ds_fosi_hi = xr.open_mfdataset(fosi_hi_files, combine="by_coords", preprocess=crop_fosi_hi,
                                   data_vars="minimal", compat="override", coords="minimal")
ds_fosi_hi


## Regrid PIOMAS thickness onto the regional domain

PIOMAS gives sea ice volume directly too, but only as a **pan-Arctic total**
(`PIOMAS.icevolume.*.txt`, not spatially resolved) -- not usable for this regional bbox. Its
gridded thickness product (`PIOMAS.hi.*.nc`, the same file loaded in `evaluation_plots.ipynb`)
has no accompanying cell-area file available locally, so a true PIOMAS-native regional SIV
can't be computed directly either.

As an approximation, PIOMAS `hi` is regridded (nearest-neighbor, 3D-Cartesian KDTree -- same
method as `evaluation_plots.ipynb`) onto the FOSI/MESACLIP native tripole grid cropped to this
bbox, then multiplied by *that* grid's `tarea` and summed. This gives a regional PIOMAS SIV with
the same footprint and magnitude convention as MESACLIP/FOSI, but it borrows someone else's
grid-cell areas for PIOMAS's coarser (~40 km) thickness field -- an approximation, not a native
PIOMAS product.

In [ ]:
from scipy.spatial import cKDTree

PIOMAS_PATH = "/glade/campaign/cgd/ccr/yeager/OBS/seaice/PIOMAS/PIOMAS.hi.1978-2020.nc"

def to_xyz(lat_deg, lon_deg):
    lat, lon = np.deg2rad(lat_deg), np.deg2rad(lon_deg)
    return np.stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)], axis=-1)

ds_piomas = xr.open_dataset(PIOMAS_PATH)
# PIOMAS's time axis is decimal year (e.g. 1978.04) -- convert to a first-of-month date
piomas_year = np.floor(ds_piomas["time"].values).astype(int)
piomas_month = np.clip(np.round((ds_piomas["time"].values - piomas_year) * 12 + 0.5).astype(int), 1, 12)
piomas_time = np.array(
    [f"{y:04d}-{m:02d}-01" for y, m in zip(piomas_year, piomas_month)], dtype="datetime64[D]"
)

# nearest PIOMAS source cell (120 x 360 native grid) for every FOSI/MESACLIP cropped grid cell
piomas_tree = cKDTree(to_xyz(ds_piomas["lat"].values.ravel(), (ds_piomas["lon"].values % 360).ravel()))
_, piomas_idx = piomas_tree.query(to_xyz(fosi_lat2d.values.ravel(), (fosi_lon2d.values % 360).ravel()))

piomas_hi_flat = ds_piomas["hi"].values.reshape(ds_piomas["hi"].shape[0], -1)
piomas_hi_regridded = piomas_hi_flat[:, piomas_idx].reshape(-1, *fosi_lat2d.shape)

# SIV in km^3 (m^3 / 1e9), using the FOSI native grid's tarea over the same cropped cells
siv_piomas_vals = (piomas_hi_regridded * ds_fosi.tarea.values[None, :, :]).sum(axis=(1, 2)) / 1e9
siv_piomas = xr.DataArray(siv_piomas_vals, dims=["time"], coords={"time": piomas_time})

### Compute climatology over the overlapping period

Full-year overlap across all three records: PIOMAS is the limiting one (1978-2020, confirmed
12 months/year at both ends of its record) -- FOSI covers 1958-2021 and MESACLIP covers
1920-2100, both comfortably wider.

In [ ]:
YEAR_MIN, YEAR_MAX = 1978, 2020  # PIOMAS's full-year coverage is the limiting record here
import gc

# SIV = sum(hi * tarea) over the cropped domain, in km^3 (m^3 / 1e9). MESACLIP is computed
# member-by-member (mesa_hi_members / mesa_members, the per-member lists from before the concat)
# for the same reason as the SIA cell above -- keeps peak memory to ~1 member's hi+tarea instead
# of all 9 concatenated members at once.
siv_mesaclip_list = []
for i, (mid, hi_member, aice_member) in enumerate(zip(member_ids, mesa_hi_members, mesa_members)):
    print(f"MESACLIP SIV: member {mid} ({i + 1}/{len(member_ids)})...", flush=True)
    siv_member = (hi_member.hi * aice_member.tarea).sum(dim=["nj", "ni"]) / 1e9
    siv_mesaclip_list.append(siv_member.compute())
    gc.collect()
siv_mesaclip = xr.concat(siv_mesaclip_list, dim="member_id").assign_coords(member_id=member_ids)
del siv_mesaclip_list
gc.collect()

siv_fosi = ((ds_fosi_hi.hi * ds_fosi.tarea).sum(dim=["nj", "ni"]) / 1e9).compute()

def month_clim(da, year_min=YEAR_MIN, year_max=YEAR_MAX):
    sub = da.sel(time=(da.time.dt.year >= year_min) & (da.time.dt.year <= year_max))
    return sub.groupby("time.month").mean(dim="time")

siv_mesaclip_clim = month_clim(siv_mesaclip)   # (member_id, month)
siv_fosi_clim = month_clim(siv_fosi)           # (month,)
siv_piomas_clim = month_clim(siv_piomas)       # (month,)
print("done.")


## Plot SIV seasonal climatology

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
nens = len(siv_mesaclip_clim.member_id)

for ii in range(nens):
    ax.plot(siv_mesaclip_clim.month, siv_mesaclip_clim.isel(member_id=ii),
            color=PALETTE["members"], linewidth=1, label="_nolegend_", zorder=1)
ax.plot(siv_mesaclip_clim.month, siv_mesaclip_clim.mean(dim="member_id"),
        label=f"MESACLIP ensemble mean (n={nens})", color=PALETTE["MESACLIP"],
        linewidth=2.2, marker="o", markersize=5, zorder=3)
ax.plot(siv_fosi_clim.month, siv_fosi_clim, label="FOSI", color=PALETTE["FOSI"],
        linewidth=2.2, marker="o", markersize=5, zorder=3)
ax.plot(siv_piomas_clim.month, siv_piomas_clim, label="PIOMAS (regional approx.)",
        color=PALETTE["PIOMAS"], linestyle="--", linewidth=2.2, marker="o", markersize=5, zorder=3)

ax.set_xticks(range(1, 13))
ax.set_xlabel("Month")
ax.set_ylabel("Regional SIV (km$^3$)")
ax.set_title(f"Regional Sea Ice Volume Climatology ({YEAR_MIN}-{YEAR_MAX})")
ax.legend()
plt.xlim(0.9, 12.1)
plt.tight_layout()
fig.savefig(dir_out + "mesaclip_fosi_piomas_siv_climatology.png", bbox_inches="tight", dpi=220)
plt.show()
